# WasteWise-AI: Smart Waste Classification Using Deep Learning (CNN)
### Continuous Assessment 1 (CA1) - Exploratory Analysis, CNN Architecture & Training Pipeline

---

## 1. Project Introduction & Objectives

Effective waste segregation is one of the most critical challenges in modern municipal waste management and sustainable environmental engineering. Improper sorting at source contaminates recycling streams, increases landfill burden, and squanders recyclable resources.

**WasteWise-AI** is an AI-powered smart waste classification system designed to automate waste identification directly from digital photographs. The model classifies waste items into six standard benchmark categories:
1. **Cardboard** (`0`)
2. **Glass** (`1`)
3. **Metal** (`2`)
4. **Paper** (`3`)
5. **Plastic** (`4`)
6. **Trash** (`5`) - Non-recyclable / miscellaneous municipal waste

### Notebook Objectives:
1. **Data Pipeline Exploration**: Inspect the verified, stratified TrashNet dataset manifests (`70% train`, `15% val`, `15% test`).
2. **Class Distribution & Imbalance**: Visualize class proportions and examine the role of balanced class weighting.
3. **Preprocessing & Augmentation**: Understand image normalization ($[0.0, 1.0]$) and training-only data augmentation (flipping, rotation, zoom).
4. **Custom CNN Architecture**: Explore the 4-block Convolutional Neural Network with Batch Normalization, Dropout, and Global Average Pooling.
5. **Architectural Deep Dive**: Review the mathematical intuition behind Conv2D, ReLU, MaxPool, Dropout, GAP, and Softmax.
6. **Pre-Flight Smoke Test**: Validate pipeline tensor shapes, forward inference, and backpropagation on a single batch.
7. **Training & Evaluation Framework**: Provide a modular, reproducible framework for full training and test evaluation.


---
## 2. Environment Setup, Library Imports & Seed Configuration

We start by setting the working directory to the workspace root, importing all required scientific and deep learning libraries, and seeding all pseudo-random number generators for strict reproducibility.


In [ ]:
import os
import sys
import json
from pathlib import Path
from collections import Counter

# Set workspace root as current working directory so all relative paths work seamlessly
cwd = Path.cwd()
if cwd.name == "notebooks" or cwd.name == "ml":
    while cwd.name in ["notebooks", "ml"]:
        cwd = cwd.parent
    os.chdir(cwd)

if str(cwd) not in sys.path:
    sys.path.insert(0, str(cwd))

print(f"[OK] Workspace root set to: {Path.cwd()}")

# Suppress verbose TensorFlow C++ messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# Import modular pipeline functions from existing project scripts
from ml.scripts.train_cnn import (
    DEFAULT_CLASS_MAPPING,
    DEFAULT_ID_TO_CLASS,
    set_reproducible_seed,
    validate_manifest,
    load_class_weights,
    create_data_augmentation_layer,
    build_tf_dataset,
    build_custom_cnn,
)

# Fix deterministic random seeds (Seed: 42)
set_reproducible_seed(42)

print(f"[OK] TensorFlow Version: {tf.__version__}")
print(f"[OK] Keras Version:      {tf.keras.__version__ if hasattr(tf.keras, '__version__') else 'Built-in Keras'}")
print(f"[OK] NumPy Version:      {np.__version__}")
print(f"[OK] Pandas Version:     {pd.__version__}")
print(f"[OK] Matplotlib Version: {plt.matplotlib.__version__}")


---
## 3. Load and Validate Existing Manifests

Rather than re-splitting data inside this notebook (which risks split drift and data leakage), we reuse the verified manifests generated by `ml/scripts/prepare_dataset.py`:
- `ml/outputs/train_manifest.csv` (1,768 images, ~70.0%)
- `ml/outputs/val_manifest.csv` (379 images, ~15.0%)
- `ml/outputs/test_manifest.csv` (380 images, ~15.0%)
- `ml/outputs/class_weights.json`


In [ ]:
train_manifest_path = Path("ml/outputs/train_manifest.csv")
val_manifest_path   = Path("ml/outputs/val_manifest.csv")
test_manifest_path  = Path("ml/outputs/test_manifest.csv")
weights_path        = Path("ml/outputs/class_weights.json")

# Validate files exist and verify class IDs
train_df = validate_manifest(train_manifest_path, DEFAULT_CLASS_MAPPING, "Train")
val_df   = validate_manifest(val_manifest_path, DEFAULT_CLASS_MAPPING, "Validation")
test_df  = validate_manifest(test_manifest_path, DEFAULT_CLASS_MAPPING, "Test")
class_weights = load_class_weights(weights_path)

total_images = len(train_df) + len(val_df) + len(test_df)
print(f"Total Dataset Images Accounted For: {total_images}")
print(f"  * Train split:      {len(train_df):>5} images ({len(train_df)/total_images*100:.2f}%)")
print(f"  * Validation split: {len(val_df):>5} images ({len(val_df)/total_images*100:.2f}%)")
print(f"  * Test split:       {len(test_df):>5} images ({len(test_df)/total_images*100:.2f}%)")

print("\nSample rows from train_manifest.csv:")
display(train_df.head(5))


---
## 4. Visualizing Sample Images from All Six Waste Categories

Understanding visual variance is essential before modeling. The TrashNet dataset contains photographs of household items taken under consistent lighting against a white background. Below, we load two representative samples from each class to observe features like texture, reflection, transparency, and structure.


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
plt.subplots_adjust(wspace=0.15, hspace=0.3)

for col_idx, (class_name, class_id) in enumerate(DEFAULT_CLASS_MAPPING.items()):
    # Get first 2 images from train split for this class
    class_rows = train_df[train_df["class_name"] == class_name].iloc[:2]
    
    for row_idx, (_, row) in enumerate(class_rows.iterrows()):
        ax = axes[row_idx, col_idx]
        img_path = Path(row["image_path"])
        
        with Image.open(img_path) as img:
            ax.imshow(img)
            
        ax.axis("off")
        if row_idx == 0:
            ax.set_title(f"{class_name.upper()}\n(Class ID: {class_id})", fontsize=11, fontweight="bold")

plt.suptitle("WasteWise-AI: Representative Samples from All 6 TrashNet Classes", fontsize=14, y=1.03, fontweight="bold")
plt.show()


---
## 5. Class Distribution Analysis & Imbalance Mitigation

A primary challenge in waste classification datasets is **class imbalance**. Some waste streams (like paper and cardboard) are abundant in everyday packaging, whereas non-recyclable miscellaneous **trash** is comparatively scarce.

### Imbalance Overview:
- **`paper`** is the largest class ($594$ total, $416$ in train, representing **23.5%** of the dataset).
- **`trash`** is the minority class ($137$ total, $96$ in train, representing only **5.4%** of the dataset).
- **Imbalance Ratio**: $416 / 96 \approx 4.34\times$.

### Mitigation Strategy: Balanced Class Weighting
If trained without adjustments, a standard cross-entropy loss would bias the network toward predicting majority classes (paper/glass) while ignoring minority classes. We compute balanced weights on the training split:

$$w_j = \frac{N_{\text{train}}}{C \times N_j}$$

Where:
- $N_{\text{train}} = 1768$ (total training samples)
- $C = 6$ (number of classes)
- $N_j$ = training samples in class $j$


In [ ]:
# Aggregate class counts per split
split_data = []
for split_name, df_split in [("Train (70%)", train_df), ("Val (15%)", val_df), ("Test (15%)", test_df)]:
    for cls in DEFAULT_CLASS_MAPPING.keys():
        cnt = (df_split["class_name"] == cls).sum()
        split_data.append({"Split": split_name, "Class": cls, "Count": cnt})

plot_df = pd.DataFrame(split_data)

plt.figure(figsize=(14, 5))
palette = ["#1f77b4", "#aec7e8", "#ff7f0e"]

ax = sns.barplot(
    data=plot_df,
    x="Class",
    y="Count",
    hue="Split",
    palette="Blues_r"
)

plt.title("WasteWise-AI: Stratified Class Distribution Across Splits", fontsize=13, fontweight="bold")
plt.xlabel("Waste Category", fontsize=11)
plt.ylabel("Number of Images", fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.legend(title="Split")

# Annotate counts on top of bars
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{int(height)}",
                    (p.get_x() + p.get_width() / 2., height),
                    ha="center", va="bottom", fontsize=8, rotation=0, xytext=(0, 2),
                    textcoords="offset points")

plt.tight_layout()
plt.show()

# Display computed class weights
print("\nComputed Balanced Class Weights (from ml/outputs/class_weights.json):")
weights_table = []
for cls_name, cls_id in DEFAULT_CLASS_MAPPING.items():
    weights_table.append({
        "Class ID": cls_id,
        "Class Name": cls_name,
        "Train Count": (train_df["class_name"] == cls_name).sum(),
        "Class Weight": round(class_weights[cls_id], 4)
    })
display(pd.DataFrame(weights_table))


---
## 6. Image Resizing, Normalization & Training-Only Augmentation

### Pipeline Architecture:
1. **Dynamic Decoding**: JPEG streams are decoded on-the-fly to RGB tensors using `tf.io.decode_jpeg`.
2. **Resizing ($224 \times 224$)**: Standardizes spatial dimensions for the CNN using bilinear interpolation.
3. **Normalization ($[0.0, 1.0]$)**: Pixel values are scaled by dividing by $255.0$. Normalization centers inputs, stabilizes weight updates, and prevents early saturation of activation functions.
4. **Data Augmentation**:
   - **Horizontal Flipping** (`RandomFlip("horizontal")`): Invariant to horizontal orientation.
   - **Subtle Rotation** (`RandomRotation(0.08)`): Rotates up to $\approx \pm 30^\circ$ to handle tilted items.
   - **Subtle Zoom** (`RandomZoom(0.08)`): Simulates variations in object camera distance.
   - **CRITICAL**: Augmentation is strictly applied **only to the training split**. Validation and test images are kept untransformed for unbiased evaluation.


In [ ]:
aug_layer = create_data_augmentation_layer()

# Select one sample image from cardboard class
sample_row = train_df[train_df["class_name"] == "cardboard"].iloc[0]
sample_img = tf.keras.utils.load_img(sample_row["image_path"], target_size=(224, 224))
sample_arr = tf.keras.utils.img_to_array(sample_img) / 255.0
sample_tensor = tf.expand_dims(sample_arr, 0)

# Display original and 3 augmented variants
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(sample_arr)
axes[0].set_title("Original (224x224)", fontsize=11, fontweight="bold")
axes[0].axis("off")

for i in range(1, 4):
    augmented_tensor = aug_layer(sample_tensor, training=True)
    axes[i].imshow(augmented_tensor[0].numpy())
    axes[i].set_title(f"Augmented Variant {i}", fontsize=11)
    axes[i].axis("off")

plt.suptitle("Training-Only Data Augmentation (Flip, Rotation, Zoom)", fontsize=13, y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 7. Building the Custom CNN Architecture

We instantiate `WasteWise_Custom_CNN`, constructed using Keras Functional API.
The network processes $224 \times 224 \times 3$ inputs through 4 progressive feature extraction blocks before global spatial pooling.


In [ ]:
# Instantiate custom CNN
model = build_custom_cnn(input_shape=(224, 224, 3), num_classes=6)

# Compile using Adam optimizer and Sparse Categorical Cross-Entropy
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"],
)

model.summary()


---
## 8. Deep Dive into Core CNN Building Blocks

For our CA1 Deep Learning report, each layer plays a distinct mathematical and operational role:

### 1. Convolution (`Conv2D`)
- **Mechanism**: A set of learnable $3 \times 3$ kernels slides across input feature maps, computing dot products between weights and local spatial patches.
- **Function**: Extracts translation-invariant spatial features (edges in early layers, object parts in deeper layers).

### 2. Rectified Linear Unit (`ReLU`) Activation
- **Formula**: $f(x) = \max(0, x)$
- **Function**: Introduces non-linearity, allowing the network to learn complex multi-modal decision boundaries while preventing vanishing gradients during backpropagation.

### 3. Batch Normalization (`BatchNormalization`)
- **Mechanism**: Normalizes layer activations across mini-batches to zero mean and unit variance, followed by learnable scale ($\gamma$) and shift ($\beta$) parameters.
- **Function**: Reduces internal covariate shift, accelerates training convergence, and acts as a mild regularizer.

### 4. Max Pooling (`MaxPooling2D`)
- **Mechanism**: Extracts the maximum activation across non-overlapping $2 \times 2$ windows, halving the spatial width and height ($224 \to 112 \to 56 \to 28 \to 14$).
- **Function**: Provides spatial translation invariance and exponentially reduces memory footprint and computational complexity.

### 5. Dropout (`Dropout`)
- **Mechanism**: Stochastically sets a fraction of neuron activations to zero during training (from $0.20$ in early blocks up to $0.50$ in the dense head).
- **Function**: Prevents neurons from co-adapting, forcing redundant representation learning and providing powerful regularization against overfitting.

### 6. Global Average Pooling (`GlobalAveragePooling2D`)
- **Mechanism**: Computes the spatial mean across the entire $14 \times 14$ activation map for each of the $256$ channels, outputting a compact $1 \times 256$ vector.
- **Why GAP over Flatten?**: Traditional flattening of a $14 \times 14 \times 256$ volume would produce $50,176$ values. Connecting that to a dense layer of $256$ units would create over **$12.8$ million weights**, causing severe overfitting on our $1,768$ image training set. GAP reduces parameters to zero for that transition while enforcing spatial invariance.

### 7. Softmax Output (`Dense(6, activation='softmax')`)
- **Formula**: $\sigma(z)_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}$
- **Function**: Transforms raw continuous logit scores into a valid probability distribution where each value is in $[0, 1]$ and the sum across all 6 classes equals exactly $1.0$.


---
## 9. Pre-Flight Pipeline & Architecture Smoke Test

Before investing computational time into 30 full training epochs, we execute a single-batch pre-flight check to verify:
1. `tf.data` pipeline yields properly shaped image batches: `(batch_size, 224, 224, 3)`.
2. Labels are integer tensors: `(batch_size,)`.
3. Pixel intensities are verified within $[0.0, 1.0]$.
4. Model forward pass outputs `(batch_size, 6)` and softmax probabilities sum to $1.0$.
5. A single optimization step (`train_on_batch`) updates weights without numerical errors.


In [ ]:
batch_size = 32
train_ds = build_tf_dataset(
    train_df,
    image_size=(224, 224),
    batch_size=batch_size,
    is_training=True,
    augmentation_layer=aug_layer,
    seed=42
)

# Fetch single batch
images, labels = next(iter(train_ds))

print("Pipeline Batch Verification:")
print(f"  * Batch Images Shape: {images.shape} (Expected: ({batch_size}, 224, 224, 3))")
print(f"  * Batch Labels Shape: {labels.shape} (Expected: ({batch_size},))")
print(f"  * Min Pixel Value:    {float(tf.reduce_min(images)):.4f}")
print(f"  * Max Pixel Value:    {float(tf.reduce_max(images)):.4f}")

# Forward pass
predictions = model(images, training=False)
print(f"\nModel Forward Pass:")
print(f"  * Prediction Shape:   {predictions.shape} (Expected: ({batch_size}, 6))")
prob_sums = np.sum(predictions.numpy(), axis=-1)
print(f"  * Softmax Check:      All sums approx 1.0 -> {np.allclose(prob_sums, 1.0, atol=1e-4)}")

# Single optimization step
loss_val, acc_val = model.train_on_batch(images, labels, class_weight=class_weights)
print(f"\nOptimization Step:")
print(f"  * Post-Step Loss:     {loss_val:.4f}")
print(f"  * Batch Accuracy:     {acc_val*100:.2f}%")
print("\n[OK] Smoke test completed successfully!")


---
## 10. Model Training Execution (Optional & Configurable)

To avoid accidental long-running jobs when opening this notebook, full training is disabled by default via `RUN_FULL_TRAINING = False`.

### Training Configuration:
- **Optimizer**: Adam ($	ext{learning rate} = 10^{-3}$)
- **Loss**: `SparseCategoricalCrossentropy`
- **Class Weights**: Active weighting for the minority `trash` class
- **Callbacks**:
  - `ModelCheckpoint`: Saves best weights to `ml/outputs/wastewise_cnn.keras` based on validation loss.
  - `EarlyStopping`: Halts if validation loss does not improve for 8 epochs.
  - `ReduceLROnPlateau`: Halves learning rate if validation loss plateaus for 3 epochs.

To train from the terminal:
```powershell
python ml/scripts/train_cnn.py --epochs 30 --batch-size 32
```


In [ ]:
# Set to True only when you explicitly want to run full training inside the notebook
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    print("[INFO] Launching full training...")
    
    val_ds = build_tf_dataset(val_df, image_size=(224, 224), batch_size=batch_size, is_training=False)
    
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath="ml/outputs/wastewise_cnn.keras",
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),
    ]
    
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )
    
    # Save training history
    history_df = pd.DataFrame(history.history)
    history_df.to_csv("ml/outputs/training_history.csv", index=False)
    print("[OK] Training complete and history saved.")
else:
    print("[INFO] Full training is skipped in this interactive session.")
    print("Execute `python ml/scripts/train_cnn.py --epochs 30 --batch-size 32` in your terminal to train.")


---
## 11. Training & Validation Diagnostic Curves

Once full training has run, `ml/outputs/training_history.csv` records the epoch-by-epoch loss and accuracy metrics. Below, we load and visualize the learning curves.


In [ ]:
history_csv_path = Path("ml/outputs/training_history.csv")

if history_csv_path.exists():
    hist_df = pd.read_csv(history_csv_path)
    epochs_range = range(1, len(hist_df) + 1)
    
    plt.figure(figsize=(14, 5))
    
    # Loss Subplot
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, hist_df["loss"], "o-", label="Train Loss", color="#1f77b4", linewidth=2)
    plt.plot(epochs_range, hist_df["val_loss"], "s--", label="Val Loss", color="#ff7f0e", linewidth=2)
    plt.title("Training vs Validation Loss", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    
    # Accuracy Subplot
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, hist_df["accuracy"], "o-", label="Train Accuracy", color="#2ca02c", linewidth=2)
    plt.plot(epochs_range, hist_df["val_accuracy"], "s--", label="Val Accuracy", color="#d62728", linewidth=2)
    plt.title("Training vs Validation Accuracy", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("[NOTE] `ml/outputs/training_history.csv` has not been generated yet.")
    print("Curves will automatically display here after running `train_cnn.py`.")


---
## 12. Unseen Test Set Evaluation & Confusion Matrix

Following standard machine learning methodology, the **test partition (380 images)** is never seen during hyperparameter tuning, model checkpointing, or training. It is evaluated only once on the best model checkpoint.

The cell below loads the actual results from `ml/outputs/test_metrics.json` and displays the saved confusion matrix heatmap.


In [ ]:
test_metrics_path = Path("ml/outputs/test_metrics.json")
cm_plot_path = Path("ml/outputs/confusion_matrix.png")

if test_metrics_path.exists():
    with open(test_metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
        
    print("=" * 70)
    print("WasteWise-AI: Final Unseen Test Set Performance")
    print("=" * 70)
    print(f"Overall Test Accuracy: {metrics['test_accuracy']*100:.2f}%")
    print(f"Overall Test Loss:     {metrics['test_loss']:.4f}")
    
    print("\nPer-Class Classification Metrics:")
    per_class_df = pd.DataFrame(metrics["per_class"]).T
    display(per_class_df)
    
    # Highlight minority class performance
    trash_stats = metrics["per_class"].get("trash", {})
    print(f"Minority Class 'trash' Analysis:")
    print(f"  * Precision: {trash_stats.get('precision', 'N/A')}")
    print(f"  * Recall:    {trash_stats.get('recall', 'N/A')}")
    print(f"  * F1-Score:  {trash_stats.get('f1_score', 'N/A')}")
    
    if cm_plot_path.exists():
        print("\nDisplaying Confusion Matrix Heatmap:")
        plt.figure(figsize=(8, 6))
        cm_img = Image.open(cm_plot_path)
        plt.imshow(cm_img)
        plt.axis("off")
        plt.title("Test Set Confusion Matrix", fontsize=13, fontweight="bold")
        plt.show()
else:
    print("[NOTE] `ml/outputs/test_metrics.json` not found yet.")
    print("Final evaluation metrics and confusion matrix will appear here after full training is completed.")


---
## 13. Limitations, Discussion & Future Roadmap

### 1. Dataset Limitations
- **Single Item vs Multi-Object Clutter**: TrashNet items are isolated and centered on plain white backgrounds. Real-world waste bins present occlusions, crumpled items, overlapping rubbish, and poor lighting.
- **Class Imbalance**: The `trash` class accounts for only 5.4% of samples. Even with balanced class weighting, more physical samples or targeted synthetic augmentation are needed to maximize recall.

### 2. Custom CNN vs Transfer Learning
- Training a custom CNN from scratch on $1,768$ images provides an excellent baseline and pedagogical demonstration of core computer vision mechanics.
- **Next Step (Phase 3)**: Benchmark this custom CNN against pre-trained transfer learning architectures (e.g., **MobileNetV2**, **EfficientNetB0**). Pre-trained ImageNet features provide robust edge, texture, and shape priors that typically boost accuracy by $15\text{--}25\%$.

### 3. Full-Stack Integration Roadmap
- **Phase 4**: Hyperparameter fine-tuning and model export to optimized `.keras` and TensorFlow Lite (`.tflite`) formats.
- **Phase 5**: Wrap inference in a FastAPI backend (`backend/app/` and `backend/services/`), returning predicted class, confidence score, and actionable recycling instructions.
- **Phase 6**: Develop a responsive frontend (`frontend/`) allowing consumers to upload photos or use their smartphone camera for instant recycling classification.
